# Sesión 3, Parte A: ensambles de árboles

**Curso: Proyecciones Macroeconómicas con Machine Learning y Deep Learning**

En la Sesión 1 vimos que un árbol aprende umbrales e interacciones pero no extrapola. Hoy la pregunta es otra: ¿cómo se convierte ese árbol inestable en los modelos que dominan el ML tabular?

## Al terminar esta parte podrás

1. explicar por qué promediar árboles (bagging) reduce varianza, y qué agrega Random Forest;
2. explicar el boosting como suma secuencial de correcciones pequeñas;
3. usar los frenos que importan: learning rate, profundidad, hojas mínimas y early stopping;
4. anticipar qué pasa con estos modelos cuando la muestra es chica (el caso macro).

## Ruta de la Parte A

| bloque | pregunta |
|---|---|
| Un árbol solo | ¿Por qué es tan inestable? |
| Bagging y Random Forest | ¿Qué gana el promedio? ¿Qué agrega decorrelacionar? |
| Boosting | ¿Qué cambia al aprender de los residuos, despacio? |
| La muestra chica | ¿Quién gana con n = 200 y quién con n = 5000? |
| Early stopping | ¿Cómo se frena el boosting sin mirar el test? |

**Pregunta guía:** bagging y boosting suman árboles; ¿por qué uno ataca la varianza y el otro el sesgo?

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor

# Localiza session3/ (utils.py + data/fredmd.csv).
candidatos = [Path.cwd(), Path.cwd() / "session3", *Path.cwd().parents]
SESSION_DIR = next(
    (
        ruta.resolve()
        for ruta in candidatos
        if (ruta / "utils.py").exists() and (ruta / "data" / "fredmd.csv").exists()
    ),
    None,
)
if SESSION_DIR is None:
    raise FileNotFoundError("No se encontró session3/data/fredmd.csv.")
if str(SESSION_DIR) not in sys.path:
    sys.path.insert(0, str(SESSION_DIR))

import utils as U

U.set_style()
RANDOM_STATE = 42
print(f"Sesión 3, Parte A | figuras en: {U.FIGS}")

## 1. Un árbol solo: flexible pero inestable

Simulamos una relación no lineal suave y ajustamos un árbol profundo sobre seis muestras bootstrap de los mismos datos. Cada muestra difiere solo por el azar del sorteo; si el modelo fuera estable, las seis curvas serían casi idénticas.

**Predicción antes de ejecutar:** ¿las seis escaleras van a contar la misma historia?

In [ ]:
import matplotlib as mpl

rng = np.random.default_rng(RANDOM_STATE)
n = 150
x = np.sort(rng.uniform(-3, 3, n))
f_verdadera = lambda x: 2.0 * np.sin(1.2 * x) + 0.4 * x
y = f_verdadera(x) + rng.normal(0, 0.6, n)
X = x.reshape(-1, 1)
grid = np.linspace(-3, 3, 500)
G = grid.reshape(-1, 1)

tonos = mpl.colormaps["Oranges"](np.linspace(0.35, 0.95, 6))
fig, ax = plt.subplots(figsize=(9.5, 4.4))
ax.scatter(x, y, s=12, color=U.MUTED, alpha=0.4, label="datos")
ax.plot(grid, f_verdadera(grid), color=U.INK, lw=1.6, ls="dashed", label="relación verdadera")
for b in range(6):
    idx = rng.integers(0, n, n)
    arbol = DecisionTreeRegressor(random_state=b).fit(X[idx], y[idx])
    ax.plot(grid, arbol.predict(G), color=tonos[b], lw=1.0, alpha=0.9,
            label="árbol por muestra bootstrap (tono = muestra 1 a 6)" if b == 0 else None)
ax.set_title("Seis muestras bootstrap, seis historias: el árbol profundo memoriza ruido", loc="left")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(fontsize=9)
U.save_fig(fig, "C01_arbol_inestable")
plt.show()

### Lectura

Cada árbol ajusta casi perfecto SU muestra y cuenta una historia distinta en los bordes y en los saltos: sesgo bajo, varianza alta. Esa varianza es exactamente lo que un promedio puede eliminar. La idea de Breiman (1996): si no podemos conseguir más datos, generemos "muestras nuevas" con bootstrap y promediemos los modelos.

## 2. Bagging: la sabiduría de promediar

Si los $B$ árboles fueran independientes con varianza $\sigma^2$, el promedio tendría varianza $\sigma^2/B$. No lo son (comparten los datos): con correlación media $\rho$ entre árboles,

$$\operatorname{Var}\left(\tfrac{1}{B}\sum_b T_b\right) = \rho\,\sigma^2 + \tfrac{1-\rho}{B}\,\sigma^2.$$

El segundo término desaparece con $B$; el primero no. Random Forest agrega el sorteo de features (`max_features`) para reducir $\rho$; cuánto sortear es un hiperparámetro más, y se elegirá con validación temporal en la Parte B.

Para ver QUÉ elimina exactamente el promedio, descomponemos el error en sesgo$^2$ + varianza sobre 100 "mundos": cien muestras nuevas del mismo DGP, y en cada una un árbol profundo y un bagging de 200 árboles.

**Predicción antes de ejecutar:** ¿el promedio reduce el sesgo, la varianza o ambos? ¿Y cuántos árboles bastan?

In [ ]:
n_mundos = 100
verdad = f_verdadera(grid)


def predicciones(modelo_factory):
    """Predicciones sobre la grilla fija, entrenando en 100 mundos del mismo DGP."""
    P = np.zeros((n_mundos, len(grid)))
    for s in range(n_mundos):
        rng_s = np.random.default_rng(1000 + s)
        x_s = rng_s.uniform(-3, 3, n)
        y_s = f_verdadera(x_s) + rng_s.normal(0, 0.6, n)
        P[s] = modelo_factory(s).fit(x_s.reshape(-1, 1), y_s).predict(G)
    return P


P_arbol = predicciones(lambda s: DecisionTreeRegressor(random_state=s))
P_bagging = predicciones(lambda s: RandomForestRegressor(
    n_estimators=200, max_features=1.0, random_state=s, n_jobs=-1))


def descompone(P):
    sesgo2 = float(((P.mean(axis=0) - verdad) ** 2).mean())
    varianza = float(P.var(axis=0).mean())
    return sesgo2, varianza


s2_arbol, var_arbol = descompone(P_arbol)
s2_bag, var_bag = descompone(P_bagging)

# ... y desde otro ángulo: el RMSE frente al número de árboles (un solo train)
Bs = [1, 2, 5, 10, 25, 50, 100, 200, 400]
rmse_B = [
    U.rmse(RandomForestRegressor(n_estimators=B, max_features=1.0,
                                 random_state=RANDOM_STATE, n_jobs=-1)
           .fit(X, y).predict(G) - verdad)
    for B in Bs
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
etiquetas = ["árbol profundo", "bagging (200 árboles)"]
axes[0].bar(etiquetas, [s2_arbol, s2_bag], color=U.TINT, edgecolor=U.BORDER, label="sesgo$^2$")
axes[0].bar(etiquetas, [var_arbol, var_bag], bottom=[s2_arbol, s2_bag],
            color=U.ACCENT, edgecolor=U.BORDER, label="varianza")
axes[0].set_ylabel("error cuadrático frente a la relación verdadera")
axes[0].set_title("Casi todo el error del árbol es varianza", loc="left", fontsize=10)
axes[0].legend(fontsize=9)

axes[1].plot(Bs, rmse_B, marker="o", color=U.OLIVE, lw=1.6)
axes[1].set_xscale("log")
axes[1].set_xlabel("número de árboles B (escala log)")
axes[1].set_ylabel("RMSE")
axes[1].set_title("La mejora se agota cerca de 100 árboles", loc="left", fontsize=10)

fig.suptitle("Bagging: qué gana el promedio y cuántos árboles bastan",
             x=0.02, ha="left", fontweight="bold")
U.save_fig(fig, "C02_bagging_rf")
plt.show()

print(f"árbol:   sesgo² = {s2_arbol:.3f} | varianza = {var_arbol:.3f}")
print(f"bagging: sesgo² = {s2_bag:.3f} | varianza = {var_bag:.3f}")
print(f"RMSE con 1 árbol: {rmse_B[0]:.3f} | con 400 árboles: {rmse_B[-1]:.3f}")

### Lectura

- El error del árbol es casi pura varianza: sesgo$^2$ de 0.004 contra varianza de 0.36.
- El bagging no toca el sesgo (ya era casi nulo) y corta la varianza a la mitad (0.36 a 0.17), solo con promediar.
- La curva de la derecha lo confirma desde otro ángulo: el error cae rápido hasta unos 100 árboles y luego se aplana. Más árboles nunca sobreajustan en un bosque; solo cuestan tiempo.

La fórmula de arriba explica el estancamiento: queda el piso $\rho\,\sigma^2$, la parte de la varianza que comparten todos los árboles por venir de los mismos datos. Para bajar ese piso hay que conseguir que los árboles se parezcan menos entre sí: Random Forest los decorrelaciona sorteando features en cada corte; el tamaño del sorteo (`max_features`) se elige en validación, como cualquier hiperparámetro.

## 3. Boosting: aprender despacio de los residuos

Bagging entrena árboles **en paralelo** sobre el mismo problema. Boosting los entrena **en secuencia**: cada árbol nuevo ajusta los residuos que dejaron los anteriores, y entra multiplicado por un learning rate $\nu$ pequeño:

$$F_M(x) = \sum_{m=1}^{M} \nu \, T_m(x), \qquad T_m \approx \text{ajuste a } y - F_{m-1}(x).$$

Árboles poco profundos (stumps de uno o dos niveles), muchos pasos, correcciones pequeñas. Friedman (2001) lo formalizó como descenso de gradiente en el espacio de funciones.

**Predicción antes de ejecutar:** con stumps de profundidad 1, ¿cuántos pasos necesita el boosting para dibujar la curva completa?

In [ ]:
gbm = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.3, max_depth=1, random_state=RANDOM_STATE,
).fit(X, y)

etapas = {1: U.GOLD, 5: U.OLIVE, 50: U.ACCENT}
fig, ax = plt.subplots(figsize=(9.5, 4.4))
ax.scatter(x, y, s=12, color=U.MUTED, alpha=0.35, label="datos")
ax.plot(grid, f_verdadera(grid), color=U.INK, lw=1.4, ls="dashed", label="relación verdadera")
for pred_parcial, m in zip(gbm.staged_predict(G), range(1, 201)):
    if m in etapas:
        ax.plot(grid, pred_parcial, color=etapas[m], lw=1.6, label=f"M = {m} árboles")
ax.set_title("El boosting dibuja la curva por capas: cada árbol corrige a los anteriores", loc="left")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(fontsize=9)
U.save_fig(fig, "C03_boosting_residuos")
plt.show()

### Lectura

Con M = 1 el modelo es un escalón; con M = 5 aparece la forma; con M = 50 la curva está completa. El learning rate reparte el aprendizaje: $\nu$ chico exige más árboles pero generaliza mejor (la pareja $\nu$ y $M$ se elige junta). A diferencia del bagging, aquí **sí** se puede sobreajustar agregando árboles: por eso el boosting necesita frenos.

XGBoost es la implementación industrial de esta idea: agrega regularización explícita en la función objetivo (un costo por hoja y un encogimiento de sus valores) y usa la curvatura de segundo orden para elegir los cortes.

## 4. [Recortable en clase] La muestra chica como protagonista

Un DGP con no linealidad genuina: efecto lineal, una interacción y un umbral, más tres features de ruido puro. Comparamos Ridge (lineal penalizado), Random Forest y XGBoost en dos mundos: $n = 200$ (tamaño macro trimestral) y $n = 5000$ (tamaño Kaggle).

**Predicción antes de ejecutar:** ¿el boosting gana en ambos mundos?

In [ ]:
def dgp_interacciones(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, (n, 6))
    y = (
        0.6 * X[:, 0]
        + 1.5 * X[:, 0] * (X[:, 1] > 0)
        + 1.2 * np.maximum(X[:, 2] - 0.5, 0)
        + rng.normal(0, 1.0, n)
    )
    return X, y


def evalua(n_train, seed):
    X_tr, y_tr = dgp_interacciones(n_train, seed)
    X_te, y_te = dgp_interacciones(4000, seed + 1)
    modelos = {
        "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 25))),
        "RandomForest": RandomForestRegressor(
            n_estimators=300, min_samples_leaf=5, random_state=seed, n_jobs=-1),
        "XGBoost": XGBRegressor(
            n_estimators=300, learning_rate=0.05, max_depth=3,
            subsample=0.8, random_state=seed, n_jobs=4),
    }
    return {nombre: U.rmse(y_te - m.fit(X_tr, y_tr).predict(X_te))
            for nombre, m in modelos.items()}


res = pd.DataFrame({
    "n = 200 (macro)": pd.Series(evalua(200, RANDOM_STATE)),
    "n = 5000 (Kaggle)": pd.Series(evalua(5000, RANDOM_STATE)),
}).round(3)
display(res)

fig, ax = plt.subplots(figsize=(8.5, 3.8))
res.T.plot.bar(ax=ax, color=[U.BLUE, U.OLIVE, U.ACCENT], edgecolor=U.BORDER, rot=0)
ax.axhline(1.0, color=U.MUTED, lw=0.9, ls="dashed")
ax.text(1.35, 1.02, "piso: DE del ruido", fontsize=8.5, color=U.MUTED)
ax.set_ylabel("RMSE fuera de muestra")
ax.set_title("La no linealidad paga solo cuando hay datos para aprenderla", loc="left")
ax.legend(fontsize=9)
U.save_fig(fig, "C04_dgp_interacciones")
plt.show()

### Lectura

Con $n = 5000$ los árboles explotan la interacción y el umbral, y el lineal queda atrás: la forma verdadera no es una recta. Con $n = 200$ la brecha se achica o se invierte: no hay suficientes datos para localizar las interacciones sin memorizar ruido, y la penalización lineal compite de igual a igual. La macro trimestral vive en el mundo de $n$ chico: este resultado, repetido una y otra vez en la literatura (Goulet Coulombe et al. 2022; Medeiros et al. 2021), es la expectativa honesta para la Parte B.

## 5. Early stopping con validación temporal

El freno más elegante del boosting: separar un bloque de validación **temporal** (el tramo final del train, nunca aleatorio), monitorear el error tras cada árbol y detenerse cuando deja de mejorar. Dos reglas de la casa:

1. el bloque de validación respeta el orden del tiempo (nada de barajar);
2. el test sigue bloqueado: early stopping es tuning, y el tuning vive en development.

**Predicción antes de ejecutar:** ¿el learning rate alto llega más lejos o se estrella antes? ¿Y qué pasa si damos más profundidad a cada árbol?

In [ ]:
rng = np.random.default_rng(7)
n = 300
t = np.arange(n)
serie = np.zeros(n)
for i in range(1, n):
    serie[i] = 0.7 * serie[i - 1] + rng.normal(0, 1)
z1 = np.roll(serie, 1); z2 = np.roll(serie, 2); z3 = np.roll(serie, 3)
objetivo = 0.5 * z1 + 1.2 * np.maximum(z2, 0) - 0.4 * z3 + rng.normal(0, 0.8, n)
X_ts = np.column_stack([z1, z2, z3])[3:]
y_ts = objetivo[3:]

corte = int(len(y_ts) * 0.75)     # bloque temporal: el último 25% valida
X_tr, X_va = X_ts[:corte], X_ts[corte:]
y_tr, y_va = y_ts[:corte], y_ts[corte:]


def curva_validacion(lr, max_depth):
    """Entrena con early stopping temporal y devuelve el RMSE de validación por iteración."""
    xgb = XGBRegressor(
        n_estimators=1500, learning_rate=lr, max_depth=max_depth,
        early_stopping_rounds=100, eval_metric="rmse",
        random_state=RANDOM_STATE, n_jobs=4,
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return xgb.evals_result()["validation_0"]["rmse"], xgb.best_iteration


fig, axes = plt.subplots(1, 2, figsize=(11, 4.0), sharey=True)

for lr, color in [(0.02, U.OLIVE), (0.3, U.ACCENT)]:
    curva, mejor = curva_validacion(lr, max_depth=2)
    axes[0].plot(curva, color=color, lw=1.5, label=f"lr = {lr} (para en {mejor})")
    axes[0].axvline(mejor, color=color, lw=0.9, ls="dashed")
axes[0].set_title("El learning rate decide cuántos árboles hacen falta", loc="left", fontsize=10)
axes[0].set_xlabel("árboles")
axes[0].set_ylabel("RMSE de validación")
axes[0].legend(fontsize=9)

for prof, color in [(2, U.OLIVE), (6, U.ACCENT)]:
    curva, mejor = curva_validacion(0.05, max_depth=prof)
    axes[1].plot(curva, color=color, lw=1.5, label=f"max_depth = {prof} (para en {mejor})")
    axes[1].axvline(mejor, color=color, lw=0.9, ls="dashed")
axes[1].set_title("Más profundidad: sobreajusta antes y llega más alto", loc="left", fontsize=10)
axes[1].set_xlabel("árboles")
axes[1].legend(fontsize=9)

fig.suptitle("Early stopping: el freno se decide en validación temporal, nunca en el test",
             x=0.02, ha="left", fontweight="bold")
U.save_fig(fig, "C05_early_stopping")
plt.show()

## 6. Síntesis: los dos ensambles frente a frente

| dimensión | Random Forest | XGBoost |
|---|---|---|
| Cómo suma árboles | en paralelo (bootstrap) | en secuencia (residuos) |
| Ataca sobre todo | la varianza | el sesgo |
| Crecimiento del árbol | profundo | poco profundo |
| Frenos críticos | `max_features`, hojas mínimas | `learning_rate`, `max_depth` |
| ¿Más árboles sobreajusta? | no (solo cuesta tiempo) | sí: usar early stopping |
| Riesgo con n chico | moderado | alto sin frenos |
| Velocidad | media | alta |

> **Conclusión:** los dos heredan las virtudes del árbol (umbrales, interacciones, robustez a escalas) y su límite (no extrapolan fuera del soporte). La elección práctica se hace con validación temporal, no por fe.

## Comprobación antes de la pausa

1. ¿Por qué el error del bagging se estanca en vez de caer a cero con más árboles?
2. ¿Qué hace exactamente `max_features` y por qué ayuda más cuando hay muchas features correlacionadas?
3. ¿Por qué el boosting puede sobreajustar con más árboles y el bagging no?
4. Si el bloque de validación del early stopping se eligiera al azar (barajando), ¿qué regla del curso estaríamos rompiendo?

# Transición a la Parte B

Teoría lista. Ahora el examen real: FRED-MD, ~250 series mensuales de EE.UU., y una pregunta de policymaker: ¿estos modelos pronostican mejor la inflación que un AR(4)? Notebook: `s3_B_fredmd.ipynb`.